## 1. 설정
수신기 와이파이 연결과 각 단계의 값. 바꿀 게 있으면 여기만 고친다.

In [ ]:
import csv
import re
import socket
import struct
import subprocess
import threading
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
import ipywidgets as widgets
from ipyevents import Event
from scipy.signal import butter, sosfiltfilt, welch

# 수신기 연결: 수신기가 연 와이파이(AP)에 맥이 접속하고 UDP로 CSI를 받음 (receiver main.c 와 같게)
AP_SSID, AP_PASS, UDP_PORT = "csi-recv", "csisensing", 5005
HDR = struct.Struct("<IIIbBH")   # CSI 한 건 앞부분: seq, lost, ts, rssi, fwi, len (16B). 뒤에 CSI 256B → 한 건 272B
REC_SIZE = HDR.size + 256
WIFI_DEV = re.search(r"Wi-Fi\nDevice: (\S+)", subprocess.run(["networksetup", "-listallhardwareports"], capture_output=True, text=True).stdout)[1]   # 맥 와이파이 장치 (보통 en0)

# 128쌍 중 쓰는 자리. 가운데·양끝 빈칸과 망가진 L-LTF +1번 제외
LLTF = np.r_[2:27, 38:64]       # 51개
HTLTF = np.r_[65:93, 100:128]   # 56개
SLOTS = np.r_[LLTF, HTLTF]      # 107개

# ① 튀는 값 제거 (가운데 + 앞뒤 2개씩 = 5개)
SPIKE_K = 0.5   # 가운데 값이 5개 중앙값과 그 중앙값의 50% 넘게 차이 나면 튄 값

# ③④⑤ 1분 묶음 처리
BLOCK_S, OVERLAP_S = 60, 15   # 60초마다 갱신, 직전 15초 겹침 → 75초씩 처리
LOW_HZ, HIGH_HZ = 0.1, 0.5   # 호흡 주기 2~10초만 남김 → 0.5Hz 위는 숨마다 반복 안 되는 잔물결이라 자름
N_TOP = 10
EDGE_S = 5.0   # 필터 가장자리라 튀는 묶음 앞뒤 몇 초. combine 에서 잘라내고 가운데만 씀

## 수신 (선언)
수신기 와이파이(csi-recv)에 접속하고 UDP 묶음(5건)을 받아, 건마다 진폭 107개 → 시각·라벨 기록 → ①② 처리.
- 접속이 안 되거나 끊기면 될 때까지 다시 접속
- 연결 상태와 패킷이 들어오는지는 실행 셀의 상태 줄에 표시

In [ ]:
SPIN = "⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏"   # 패킷 들어오는 동안 돌아가는 글자


def parse(d, k):
    """UDP 묶음 d의 k번째 CSI → (ts, 107개 세기, rssi). 길이가 256이 아니면 None"""
    seq, lost, ts, rssi, fwi, ln = HDR.unpack_from(d, k * REC_SIZE)
    if ln != 256:
        return None
    pairs = np.frombuffer(d, np.int8, 256, k * REC_SIZE + HDR.size).astype(float).reshape(128, 2)[SLOTS]
    return ts, np.hypot(pairs[:, 0], pairs[:, 1]), rssi


def my_ip():
    return subprocess.run(["ipconfig", "getifaddr", WIFI_DEV], capture_output=True, text=True).stdout.strip()


def join():
    """수신기 와이파이 접속 시도. 수신기가 준 IP(192.168.4.x)를 받으면 True (최대 약 10초)"""
    try:
        subprocess.run(["networksetup", "-setairportnetwork", WIFI_DEV, AP_SSID, AP_PASS], capture_output=True, timeout=15)
    except subprocess.TimeoutExpired:
        return False
    for _ in range(20):
        if my_ip().startswith("192.168.4."):
            return True
        time.sleep(0.5)
    return False


def rejoin_home():
    """수신기 와이파이에서 나와 원래 쓰던 와이파이로 돌아감: 맥에 저장된 와이파이를 순서대로 시도 (비밀번호는 키체인). 붙은 이름, 없으면 None"""
    saved = subprocess.run(["networksetup", "-listpreferredwirelessnetworks", WIFI_DEV], capture_output=True, text=True).stdout.splitlines()[1:]
    for ssid in [x.strip() for x in saved if x.strip() and x.strip() != AP_SSID]:
        try:
            r = subprocess.run(["networksetup", "-setairportnetwork", WIFI_DEV, ssid], capture_output=True, text=True, timeout=15)
        except subprocess.TimeoutExpired:             # 키체인 창 등으로 15초 넘게 멈추면 다음 와이파이로
            continue
        if "Could not find" in r.stdout + r.stderr:   # 근처에 없는 와이파이는 바로 다음으로
            continue
        for _ in range(20):                           # IP 받을 때까지 최대 10초
            ip = my_ip()
            if ip and not ip.startswith("192.168.4."):
                return ssid
            time.sleep(0.5)
    return None


def reader():
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.bind(("", UDP_PORT))   # 모든 주소로 받음 → 재접속해서 IP가 바뀌어도 그대로
    sock.settimeout(1)
    prev, t, connected, tries, shown, spin, rssi = None, 0.0, False, 0, 0.0, 0, None
    try:
        while not stop.is_set():
            if not connected:   # 접속 안 됨 / 끊김 → 될 때까지 다시 시도
                tries += 1
                net.value = f"{AP_SSID} 접속 시도 {tries}번째 (지금 IP {my_ip() or '없음'})"
                connected = join()
                if connected:
                    net.value = f"{AP_SSID} 연결됨, IP {my_ip()}"
                continue
            try:
                d, _ = sock.recvfrom(4096)
            except socket.timeout:   # 1초 넘게 안 옴: 끊겼는지 확인
                connected = my_ip().startswith("192.168.4.")
                if connected:
                    net.value = f"{AP_SSID} 연결됨 · 패킷 안 들어옴"
                continue
            for k in range(len(d) // REC_SIZE):
                p = parse(d, k)
                if p is None:
                    continue
                ts, amp, rssi = p
                if prev is not None:
                    t += ((ts - prev) % 2**32) / 1e6   # ESP32 시계 한 바퀴 넘어가도 이어짐
                prev = ts
                t_list.append(t)
                label_list.append(label)
                c = despike_push(amp)   # ①: 2패킷 전 줄이 정리돼서 나옴 (처음 4개는 None)
                if c is not None:
                    tc = t_list[-3]   # 그 줄의 시각
                    clean_t.append(tc)   # 시각 먼저: len(clean_list) 까지는 시각이 다 있음
                    clean_list.append(scale_row(c))   # ②: 패킷 공통 흔들림 제거
                amp_list.append(amp)   # 마지막에 추가: len(amp_list) 까지는 시각·라벨·세기가 다 채워져 있음
            if time.time() >= shown + 0.15:   # 패킷이 들어오는 동안 글자가 돌아감 (안 들어오면 멈춤) + 마지막 CSI의 RSSI
                net.value = f"<code>{SPIN[spin % len(SPIN)]}</code> {AP_SSID} 연결됨 · 패킷 들어오는 중 · RSSI {rssi} dBm"
                shown, spin = time.time(), spin + 1
    finally:
        sock.close()

## ① 튀는 값 제거 (선언)
가운데 값을 앞뒤 2개씩 포함한 5개의 중앙값과 비교해 현저히 차이 나면 그 중앙값으로 바꾼다. 중앙값이라 튄 값이 2개까지 섞여도 기준이 안 흔들림. 2패킷(약 0.02초) 늦음.

In [ ]:
win = []   # 최근 5줄 (가운데 = 판단할 줄)


def despike_push(amp):
    """새 줄을 넣고, 가운데(2개 전) 줄이 5줄 중앙값과 현저히 다르면 그 중앙값으로 바꿔 돌려준다. 5줄이 안 찼으면 None"""
    win.append(amp)
    if len(win) > 5:
        win.pop(0)
    if len(win) < 5:
        return None
    x = win[2].copy()
    m = np.median(win, axis=0)   # 5줄(가운데 포함) 중앙값, 서브캐리어별
    bad = np.abs(x - m) > SPIKE_K * m
    x[bad] = m[bad]
    return x

## ② 스케일 변동 제거 (선언)
패킷마다 107개 중앙값으로 나눠 패킷 공통 흔들림을 지운다. (뒤에서 줄기마다 z-score 하므로 크기를 되돌릴 필요 없음)

In [ ]:
def scale_row(x):
    """튀는 값 정리된 107개 한 줄 → 패킷 공통 흔들림을 지운 한 줄"""
    return x / np.median(x)   # 평균보다 호흡에 크게 흔들리는 줄에 덜 끌려감

## ③④⑤ 1분 묶음 처리 (선언)
①② 결과가 1분 쌓일 때마다 직전 15초를 겹쳐 75초로: 0.1~0.5Hz 필터(앞뒤) → 튀는 앞뒤 5초 잘라내고 가운데만 → 줄기 점수 → 상위 N개 합치기 → 앞 묶음과 겹친 5초에서 서서히 섞어 이어 붙이기

In [ ]:
final_t, final_y = np.empty(0), np.empty(0)   # 이어 붙인 최종 파형
blocks = []                                    # 묶음마다 고른 줄기 기록


def combine(t, x):
    """①② 결과 한 묶음 (시각, (패킷 수, 107)) → 앞뒤 5초 뺀 시각, 최종 파형 1개, 고른 줄기 정보"""
    n = len(x)
    fs = (n - 1) / (t[-1] - t[0])
    sos = butter(2, [LOW_HZ, HIGH_HZ], btype="bandpass", fs=fs, output="sos")
    f = sosfiltfilt(sos, x, axis=0, padtype="even", padlen=min(n - 1, int(10 * fs)))   # ③④ 앞뒤로 두 번
    e = int(EDGE_S * fs)
    t, f = t[e:n - e], f[e:n - e]                                      # 필터 가장자리(앞뒤 5초)는 튀므로 여기서 버리고, 이후는 가운데만
    F, P = welch(f, fs=fs, nperseg=min(len(f), int(30 * fs)), axis=0)
    band = (F >= LOW_HZ) & (F <= HIGH_HZ)
    peak = F[band][np.argmax(P[band], axis=0)]
    total = P[band].sum(axis=0)
    rhythm = np.array([P[band & (np.abs(F - peak[k]) <= 0.05), k].sum() for k in range(f.shape[1])]) / total   # 분자도 대역 안만 → 0~1
    z = (f - f.mean(axis=0)) / f.std(axis=0)
    cc = np.corrcoef(z.T)
    np.fill_diagonal(cc, np.nan)
    score = rhythm * np.nanmean(np.abs(cc), axis=1)
    top = np.argsort(-score)[:N_TOP]
    sign = np.sign([np.corrcoef(z[:, top[0]], z[:, k])[0, 1] for k in top])
    weight = score[top] / score[top].sum()
    return t, (z[:, top] * sign * weight).sum(axis=1), dict(top=top.tolist(), sign=sign.tolist(), score=score[top].tolist())


def add_block(t, y):
    """새 묶음(가장자리 이미 뺀 것)을 이어 붙임. 크기 1로 맞추고, 앞 묶음과 겹친 구간에서 방향 맞춘 뒤 서서히 섞음"""
    global final_t, final_y
    y = y / y.std()
    if len(final_t) == 0:
        final_t, final_y = t, y
        return
    n = int((t <= final_t[-1]).sum())              # 겹친 구간 (겹침 15초 - 양쪽 가장자리 5초씩 = 5초)
    if n > 1:
        prev = final_y[-n:]
        if np.corrcoef(prev, y[:n])[0, 1] < 0:
            y = -y
        w = np.linspace(0, 1, n)                   # 겹친 구간에서 앞 묶음 → 새 묶음으로 서서히
        final_y[-n:] = (1 - w) * prev + w * y[:n]
    final_t = np.concatenate([final_t, t[n:]])
    final_y = np.concatenate([final_y, y[n:]])


def process_until(end):
    """①② 결과가 end초까지 쌓였으면 [end - BLOCK_S - OVERLAP_S, 지금] 묶음을 처리. 처리했으면 True"""
    n = len(clean_list)
    if not n or clean_t[n - 1] < end:
        return False
    t = np.array(clean_t[:n])
    use = t >= end - BLOCK_S - OVERLAP_S
    tb, y, info = combine(t[use], np.array(clean_list[:n])[use])
    add_block(tb, y)
    blocks.append(dict(start_s=float(tb[0]), end_s=float(tb[-1]), **info))
    return True


def blocker():
    end = BLOCK_S
    while not stop.is_set():
        if process_until(end):
            end += BLOCK_S
        time.sleep(1)

## 화면 (선언)
최종 호흡 파형은 1분 묶음이 새로 붙을 때마다 다시 그린다.
최종 파형에는 기준 파형(키 라벨 → 지금은 키보드, 나중에 호흡 벨트)을 같은 필터·크기로 맞춰 겹쳐 그린다.

In [ ]:
def ref_wave():
    """기준 파형: 키 라벨을 가슴 위치처럼 바꿈 (in 누르는 동안 올라감, ex 동안 내려감, none 멈춤)
    → 최종 파형과 같은 0.1~0.5Hz 필터, 크기 1. 나중에 호흡 벨트를 달면 이 함수만 벨트 신호로 바꾸면 됨"""
    n = len(t_list)
    tt, lab = np.array(t_list[:n]), np.array(label_list[:n])
    v = np.where(lab == "in", 1.0, np.where(lab == "ex", -1.0, 0.0))
    pos = np.concatenate([[0.0], np.cumsum(v[1:] * np.diff(tt))])   # 누른 시간만큼 오르내림
    fs = (n - 1) / (tt[-1] - tt[0])
    sos = butter(2, [LOW_HZ, HIGH_HZ], btype="bandpass", fs=fs, output="sos")
    r = sosfiltfilt(sos, pos, padtype="even", padlen=min(n - 1, int(10 * fs)))
    sd = r.std()
    return tt, r / sd if sd > 0 else r


def match_sign(t, y, tr, r):
    """화면용: 최종 파형 위아래를 기준 파형과 맞춤 (CSI 파형은 어느 쪽이 들숨인지 정해져 있지 않음)"""
    ri = np.interp(t, tr, r)
    return -y if ri.std() > 0 and np.corrcoef(y, ri)[0, 1] < 0 else y


def draw_final():
    """이어 붙인 최종 파형 + 기준 파형(키 라벨)을 겹쳐 다시 그림"""
    t, y = final_t, final_y
    tr, r = ref_wave()
    y = match_sign(t, y, tr, r)
    m = (tr >= t[0]) & (tr <= t[-1])
    ax_w.clear()
    ax_w.plot(tr[m], r[m], color="tab:orange", lw=1.1, label="reference (key)")
    ax_w.plot(t, y, "k", lw=1.1, label="CSI")
    ax_w.legend(loc="upper right")
    ax_w.grid(alpha=0.3)
    ax_w.set(xlim=(t[0], t[-1]), xlabel="time (s)")
    ax_w.set_title(f"breathing waveform (all steps) vs reference  {len(blocks)} blocks   CSI sign matched to reference", loc="left")
    view_w.update(fig_w)


def show_final():
    drawn = 0                                # 이미 그린 묶음 수
    while not stop.is_set():
        if len(blocks) > drawn and len(final_t):
            drawn = len(blocks)
            draw_final()
        time.sleep(5)


def save_waveform_rows(path, row_s=50):
    """저장용: 최종 파형과 기준 파형을 row_s초씩 여러 줄로 나눠 겹쳐 크게 그림 (화면용 그래프와 별개)"""
    from matplotlib.figure import Figure
    t, y = final_t, final_y
    tr, r = ref_wave()
    y = match_sign(t, y, tr, r)
    nrow = max(1, int(np.ceil((t[-1] - t[0]) / row_s)))
    fig = Figure(figsize=(20, 3.2 * nrow), layout="constrained")
    axes = np.atleast_1d(fig.subplots(nrow, 1))
    lim = max(np.percentile(np.abs(y), 99.5), np.percentile(np.abs(r), 99.5)) * 1.1
    for j, ax in enumerate(axes):
        a0 = t[0] + j * row_s
        a1 = a0 + row_s
        mr = (tr >= a0) & (tr <= a1)
        m = (t >= a0) & (t <= a1)
        ax.plot(tr[mr], r[mr], color="tab:orange", lw=1.2, label="reference (key)")
        ax.plot(t[m], y[m], "k", lw=1.2, label="CSI")
        ax.set_xlim(a0, a1)
        ax.set_ylim(-lim, lim)
        ax.grid(alpha=0.3)
    axes[0].legend(loc="upper right")
    axes[0].set_title(f"{run_dir.name}  breathing waveform (filter {LOW_HZ}-{HIGH_HZ} Hz) vs reference   CSI sign matched to reference",
                      loc="left", fontsize=12)
    axes[-1].set_xlabel("time (s)")
    fig.savefig(path, dpi=80)

## 실행
**커널 재시작 후 위 셀들(설정, 파이프라인 선언)을 실행한 다음, 이 셀을 한 번만** 실행.
누르는 순간 수신기 와이파이(csi-recv)에 접속해서 수집이 시작되고 (끊기면 알아서 다시 접속), 아래에 라벨 상자 · 연결 상태(패킷이 들어오는지) · 1분마다 갱신되는 최종 호흡 파형이 뜬다.
라벨: 마우스를 라벨 상자 위에 둔 채로 ← 누르고 있기 = in(들숨), → 누르고 있기 = ex(날숨), 안 누르면 none
시작하면 `날짜_시각/` 측정 폴더가 바로 만들어진다.

In [ ]:
# ===== 기록 목록 + 정답 라벨 + 연결 상태 =====
t_list, label_list, amp_list = [], [], []   # 패킷마다 한 줄씩 누적 (시각, 라벨, 107개 세기)
clean_list, clean_t = [], []   # ① 튀는 값 + ② 스케일 변동 제거 결과와 그 줄의 시각 (2패킷 늦게 채워짐)
stop = threading.Event()
label = "none"   # 지금 눌린 키의 라벨. 패킷이 들어올 때 이 값을 같이 기록

# 키보드 입력: 누르는 동안 in/ex, 떼면 none
KEYMAP = {"ArrowLeft": "in", "ArrowRight": "ex"}
pad = widgets.HTML(layout=widgets.Layout(width="420px", height="60px", border="2px solid gray"))
pad.value = "<h3>&nbsp;label: none</h3>"
net = widgets.HTML("연결 준비 중")   # 수신기 와이파이 연결 상태 + 패킷이 들어오는지


def on_key(e):
    global label
    if e["key"] in KEYMAP:
        label = KEYMAP[e["key"]] if e["type"] == "keydown" else "none"
        pad.value = f"<h3>&nbsp;label: {label}</h3>"


keys = Event(source=pad, watched_events=["keydown", "keyup"], prevent_default_action=True)
keys.on_dom_event(on_key)


# ===== 측정 폴더: 시작할 때 만들고, 종료할 때 저장 =====
run_dir = Path(f"{datetime.now():%Y%m%d_%H%M%S}")
run_dir.mkdir()


# ===== 화면 띄우고 시작 =====
display(pad, net)   # 라벨 상자, 연결 상태

# 화면: 1분마다 갱신되는 최종 호흡 파형 (+ 기준 파형)
fig_w, ax_w = plt.subplots(figsize=(14, 4), layout="constrained")
view_w = display(fig_w, display_id=True)
plt.close(fig_w)

# 읽기 + 1분 묶음 처리 + 파형 그리기를 뒤에서 시작
for target in (reader, blocker, show_final):
    threading.Thread(target=target, daemon=True).start()

## 종료
멈추고 소켓을 닫고 원래 와이파이로 다시 연결한 뒤 (1분이 안 찬 마지막 부분은 버림), 측정 폴더(`날짜_시각/`, 실행할 때 만든 것)에 저장
- amplitude.csv    : 시각, 키보드 라벨, 107개 진폭
- waveform.png     : 최종 호흡 파형 그래프 (①②③④⑤ 다 거친 것)

In [24]:
stop.set()
time.sleep(1.5)   # 읽기 스레드가 멈추고 소켓을 닫을 때까지 (UDP 대기 최대 1초)

# 1분 묶음이 덜 찬 마지막 부분은 처리하지 않고 버림 (원본 진폭은 amplitude.csv에 다 저장)
if len(final_t):   # 화면 스레드는 5초마다 확인하니, 마지막 묶음을 아직 못 그렸을 수 있어 한 번 더 그림
    draw_final()


# 1) 진폭 + 키보드 라벨
n = len(amp_list)
with open(run_dir / "amplitude.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["time_s", "label", *[f"c{i}" for i in range(len(SLOTS))]])
    for t, lab, row in zip(t_list[:n], label_list[:n], amp_list[:n]):
        w.writerow([f"{t:.6f}", lab, *[f"{v:.6g}" for v in row]])

# 2) 최종 호흡 파형 그래프 (①②③④⑤ 다 거친 것)
if len(final_t):
    save_waveform_rows(run_dir / "waveform.png")   # 50초씩 여러 줄로 크게

# 3) 저장이 끝난 뒤 원래 와이파이로 (여기서 오래 걸려도 파일은 이미 저장됨)
print("원래 와이파이:", rejoin_home() or "다시 연결 못 함 (와이파이 메뉴에서 직접 연결)")

run_dir, n

원래 와이파이: SK_WiFiGIGAC550_2.4G


(PosixPath('20261124_044107'), 23145)